# Notebook 08: Warm-Start MAML — XuetangX

**Purpose:** Ablation — warm-start init WITHOUT SRS-Adaptive inner loop. Shows that warm-start alone is insufficient.

**Protocol:**
- K=5 support pairs per episode (inner-loop adaptation)
- Q=10 query pairs per episode (outer-loop evaluation)
- Test on held-out cold-start users (disjoint from train/val)

**Config differences from NB07 (CLAUDE.md):**
```
outer_lr        = 0.0001   ← LOWER to preserve pretrained knowledge
model.load_state_dict(torch.load("models/baselines/gru_global_xuetangx.pth"))
# Inner loop: standard uniform loss — NO SRS modifications
```

**All other hyperparameters same as NB07:**
```
alpha_base      = 0.01
num_inner_steps = 5
batch_size      = 32
n_iterations    = 3000
seed            = 20260107
```

**Primary metrics:** HR@10, NDCG@10  
**This is an ABLATION** — compare against NB07 (Standard MAML) and NB11 (Warm-Start + SRS-Adaptive).

In [1]:
# [CELL 08-00] Bootstrap: repo root + paths + helpers

import os
# Required for deterministic CuBLAS on CUDA >= 10.2
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import json
import time
import uuid
import math
import copy
import pickle
import hashlib
import random
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"[CELL 08-00] start={datetime.now().isoformat(timespec='seconds')}")
print("[CELL 08-00] CWD:", Path.cwd().resolve())


def find_repo_root(start: Path) -> Path:
    """Search upward for PROJECT_STATE.md — single source of truth."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "PROJECT_STATE.md").exists():
            return p
    raise RuntimeError(
        "Could not find PROJECT_STATE.md in current or parent directories. "
        "Open this notebook from inside the repo."
    )


REPO_ROOT = find_repo_root(Path.cwd())

PATHS = {
    "PROJECT_STATE":  REPO_ROOT / "PROJECT_STATE.md",
    "META_REGISTRY":  REPO_ROOT / "meta.json",
    "DATA_RAW":       REPO_ROOT / "data" / "raw",
    "DATA_INTERIM":   REPO_ROOT / "data" / "interim",
    "DATA_PROCESSED": REPO_ROOT / "data" / "processed",
    "NOTEBOOKS":      REPO_ROOT / "notebooks",
    "REPORTS":        REPO_ROOT / "reports",
    "MODELS":         REPO_ROOT / "models",
    "RUNS":           REPO_ROOT / "runs",
}

for p in PATHS.values():
    Path(p).mkdir(parents=True, exist_ok=True) if str(p).endswith(("reports", "models", "runs")) else None
(PATHS["MODELS"] / "baselines").mkdir(parents=True, exist_ok=True)
(PATHS["MODELS"] / "maml").mkdir(parents=True, exist_ok=True)
(PATHS["REPORTS"]).mkdir(parents=True, exist_ok=True)


def cell_start(cid: str, title: str, **kw) -> float:
    t = time.time()
    print(f"\n[{cid}] {title}")
    print(f"[{cid}] start={datetime.now().isoformat(timespec='seconds')}")
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    return t


def cell_end(cid: str, t0: float, **kw) -> None:
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    print(f"[{cid}] elapsed={time.time()-t0:.2f}s  done")


def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    tmp.replace(path)


def read_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing required JSON file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


print(f"[CELL 08-00] REPO_ROOT={REPO_ROOT}")
print("[CELL 08-00] done")

[CELL 08-00] start=2026-04-09T14:28:03
[CELL 08-00] CWD: /home/user/jamalla/anonymous-users-mooc-session-meta/notebooks/xuetangx
[CELL 08-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 08-00] done


In [2]:
# [CELL 08-01] Seed + global constants

t0 = cell_start("CELL 08-01", "Seed everything + global constants")

GLOBAL_SEED = 20260107
NOTEBOOK_NAME = "08_warmstart_maml_xuetangx"
DATASET = "xuetangx"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print(f"[CELL 08-01] WARN: use_deterministic_algorithms: {e}")


seed_everything(GLOBAL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cell_end("CELL 08-01", t0, seed=GLOBAL_SEED, device=DEVICE)


[CELL 08-01] Seed everything + global constants
[CELL 08-01] start=2026-04-09T14:28:03
[CELL 08-01] seed=20260107
[CELL 08-01] device=cuda
[CELL 08-01] elapsed=0.05s  done


In [3]:
# [CELL 08-02] Run config + data paths
#
# KEY DIFFERENCE FROM NB07:
#   OUTER_LR = 0.0001  ← 10x lower than NB07 (0.001)
#   This preserves pretrained knowledge loaded from gru_global_xuetangx.pth
#   Inner loop: standard uniform gradient steps — NO SRS modifications

t0 = cell_start("CELL 08-02", "Run config + paths")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS["REPORTS"] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / "report.json"
CONFIG_PATH   = OUT_DIR / "config.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

# ── Data paths ────────────────────────────────────────────────
EPISODES_DIR = PATHS["DATA_PROCESSED"] / "xuetangx" / "episodes"
PAIRS_DIR    = PATHS["DATA_PROCESSED"] / "xuetangx" / "pairs"
VOCAB_DIR    = PATHS["DATA_PROCESSED"] / "xuetangx" / "vocab"
MODELS_DIR   = PATHS["MODELS"] / "maml"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Pretrained backbone from NB06
PRETRAINED_PATH = PATHS["MODELS"] / "baselines" / f"gru_global_{DATASET}.pth"

K_SUPPORT = 5
Q_QUERY   = 10

EPISODES_TRAIN = EPISODES_DIR / f"episodes_train_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_VAL   = EPISODES_DIR / f"episodes_val_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_TEST  = EPISODES_DIR / f"episodes_test_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
PAIRS_TRAIN    = PAIRS_DIR / "pairs_train.parquet"
PAIRS_VAL      = PAIRS_DIR / "pairs_val.parquet"
PAIRS_TEST     = PAIRS_DIR / "pairs_test.parquet"
VOCAB_PATH     = VOCAB_DIR / "course2id.json"
CHECKPOINT_PATH = MODELS_DIR / f"maml_warmstart_{DATASET}.pth"
RESULTS_PATH   = OUT_DIR / f"results_{NOTEBOOK_NAME}.json"

# ── Hyperparameters ───────────────────────────────────────────
# Same as NB07 EXCEPT outer_lr is 10x lower (0.0001 vs 0.001)
ALPHA_BASE      = 0.01       # inner-loop step size (unchanged from NB07)
NUM_INNER_STEPS = 5          # inner-loop gradient steps (unchanged from NB07)
OUTER_LR        = 0.0001     # ← KEY CHANGE: lower LR to preserve pretrained init
BATCH_SIZE      = 32         # meta-batch size (unchanged from NB07)
N_ITERATIONS    = 3000       # total outer iterations (unchanged from NB07)
VAL_EVERY       = 100
MAX_SEQ_LEN     = 50

# ── GRU architecture (locked — same as NB06/NB07) ────────────
GRU_CFG = {
    "embedding_dim": 64,
    "hidden_dim":    128,
    "num_layers":    1,
    "dropout":       0.0,
}

CFG = {
    "notebook":          NOTEBOOK_NAME,
    "dataset":           DATASET,
    "global_seed":       GLOBAL_SEED,
    "device":            DEVICE,
    "K_support":         K_SUPPORT,
    "Q_query":           Q_QUERY,
    "alpha_base":        ALPHA_BASE,
    "num_inner_steps":   NUM_INNER_STEPS,
    "outer_lr":          OUTER_LR,
    "batch_size":        BATCH_SIZE,
    "n_iterations":      N_ITERATIONS,
    "val_every":         VAL_EVERY,
    "max_seq_len":       MAX_SEQ_LEN,
    "gru_cfg":           GRU_CFG,
    "pretrained_path":   str(PRETRAINED_PATH),
    "warmstart":         True,
    "srs_adaptive":      False,
    "inputs": {
        "episodes_train": str(EPISODES_TRAIN),
        "episodes_val":   str(EPISODES_VAL),
        "episodes_test":  str(EPISODES_TEST),
        "pairs_train":    str(PAIRS_TRAIN),
        "pairs_val":      str(PAIRS_VAL),
        "pairs_test":     str(PAIRS_TEST),
        "vocab":          str(VOCAB_PATH),
    },
}

write_json_atomic(CONFIG_PATH, CFG)

# ── Init report / manifest ────────────────────────────────────
write_json_atomic(REPORT_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "metrics": {}, "key_findings": [], "sanity_samples": {},
    "data_fingerprints": {}, "notes": [],
})
write_json_atomic(MANIFEST_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG, "artifacts": []
})

# ── Append to meta.json ───────────────────────────────────────
META = PATHS["META_REGISTRY"]
if not META.exists():
    write_json_atomic(META, {"schema_version": 1, "runs": []})
m = read_json(META)
m["runs"].append({"run_id": RUN_ID, "notebook": NOTEBOOK_NAME,
                   "run_tag": RUN_TAG, "out_dir": str(OUT_DIR)})
write_json_atomic(META, m)

print(f"[CELL 08-02] ABLATION: Warm-Start MAML (NO SRS-Adaptive)")
print(f"[CELL 08-02] alpha_base={ALPHA_BASE}  inner_steps={NUM_INNER_STEPS}  outer_lr={OUTER_LR}  ← LOWER than NB07")
print(f"[CELL 08-02] batch_size={BATCH_SIZE}  n_iterations={N_ITERATIONS}  val_every={VAL_EVERY}")
print(f"[CELL 08-02] pretrained_path={PRETRAINED_PATH}")
cell_end("CELL 08-02", t0, out_dir=str(OUT_DIR))


[CELL 08-02] Run config + paths
[CELL 08-02] start=2026-04-09T14:28:03
[CELL 08-02] ABLATION: Warm-Start MAML (NO SRS-Adaptive)
[CELL 08-02] alpha_base=0.01  inner_steps=5  outer_lr=0.0001  ← LOWER than NB07
[CELL 08-02] batch_size=32  n_iterations=3000  val_every=100
[CELL 08-02] pretrained_path=/home/user/jamalla/anonymous-users-mooc-session-meta/models/baselines/gru_global_xuetangx.pth
[CELL 08-02] out_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/reports/08_warmstart_maml_xuetangx/20260409_142803
[CELL 08-02] elapsed=0.02s  done


In [4]:
# [CELL 08-03] Load data: vocab, episodes, pairs

t0 = cell_start("CELL 08-03", "Load vocab + episodes + pairs")

# ── Guard: required files must exist ─────────────────────────
for label, path in [
    ("vocab",          VOCAB_PATH),
    ("episodes_train", EPISODES_TRAIN),
    ("episodes_val",   EPISODES_VAL),
    ("episodes_test",  EPISODES_TEST),
    ("pairs_train",    PAIRS_TRAIN),
    ("pairs_val",      PAIRS_VAL),
    ("pairs_test",     PAIRS_TEST),
]:
    if not Path(path).exists():
        raise RuntimeError(
            f"Missing {label}: {path}\n"
            "Run notebooks 01->02->03->03b->04->05 first."
        )

# Guard: pretrained model must exist
if not PRETRAINED_PATH.exists():
    raise RuntimeError(
        f"Missing pretrained model: {PRETRAINED_PATH}\n"
        "Run notebook 06_base_model_selection first to produce gru_global_xuetangx.pth."
    )

# ── Vocab ────────────────────────────────────────────────────
course2id = read_json(VOCAB_PATH)
id2course  = {int(v): k for k, v in course2id.items()}
n_items    = len(course2id)
print(f"[CELL 08-03] Vocabulary: {n_items} courses")

# ── Episodes ─────────────────────────────────────────────────
episodes_train = pd.read_parquet(EPISODES_TRAIN)
episodes_val   = pd.read_parquet(EPISODES_VAL)
episodes_test  = pd.read_parquet(EPISODES_TEST)

print(f"[CELL 08-03] episodes_train: {len(episodes_train):,} ({episodes_train['user_id'].nunique():,} users)")
print(f"[CELL 08-03] episodes_val:   {len(episodes_val):,} ({episodes_val['user_id'].nunique():,} users)")
print(f"[CELL 08-03] episodes_test:  {len(episodes_test):,} ({episodes_test['user_id'].nunique():,} users)")

# ── Pairs ────────────────────────────────────────────────────
pairs_train = pd.read_parquet(PAIRS_TRAIN)
pairs_val   = pd.read_parquet(PAIRS_VAL)
pairs_test  = pd.read_parquet(PAIRS_TEST)

print(f"[CELL 08-03] pairs_train: {len(pairs_train):,}")
print(f"[CELL 08-03] pairs_val:   {len(pairs_val):,}")
print(f"[CELL 08-03] pairs_test:  {len(pairs_test):,}")

# ── Index pairs by pair_id for fast episode lookup ────────────
pairs_train_idx = pairs_train.set_index("pair_id")
pairs_val_idx   = pairs_val.set_index("pair_id")
pairs_test_idx  = pairs_test.set_index("pair_id")

cell_end("CELL 08-03", t0, n_items=n_items)


[CELL 08-03] Load vocab + episodes + pairs
[CELL 08-03] start=2026-04-09T14:28:04
[CELL 08-03] Vocabulary: 1517 courses
[CELL 08-03] episodes_train: 113,920 (4,645 users)
[CELL 08-03] episodes_val:   942 (942 users)
[CELL 08-03] episodes_test:  986 (986 users)
[CELL 08-03] pairs_train: 344,532
[CELL 08-03] pairs_val:   69,663
[CELL 08-03] pairs_test:  73,501
[CELL 08-03] n_items=1517
[CELL 08-03] elapsed=0.51s  done


In [5]:
# [CELL 08-04] Evaluation metrics: HR@1/5/10, NDCG@10, MRR

t0 = cell_start("CELL 08-04", "Define evaluation metrics")


def ndcg_at_k(ranked_items: np.ndarray, true_item: int, k: int = 10) -> float:
    """NDCG@K = 1/log2(rank+2) if true_item in top-K, else 0."""
    for i, item in enumerate(ranked_items[:k]):
        if item == true_item:
            return 1.0 / math.log2(i + 2)
    return 0.0


def compute_metrics(
    scores: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    """
    Compute HR@1, HR@5, HR@10, NDCG@10, MRR.

    Args:
        scores: (n_samples, n_items) float score matrix (higher = more relevant)
        labels: (n_samples,) int true item indices

    Returns:
        dict with keys: hr1, hr5, hr10, ndcg10, mrr
    """
    n = len(labels)
    ranked = np.argsort(-scores, axis=1)  # descending rank

    hr1_vals    = []
    hr5_vals    = []
    hr10_vals   = []
    ndcg10_vals = []
    mrr_vals    = []

    for i in range(n):
        true = int(labels[i])
        rank_arr = ranked[i]

        hr1_vals.append(1.0 if rank_arr[0] == true else 0.0)
        hr5_vals.append(1.0 if true in rank_arr[:5] else 0.0)
        hr10_vals.append(1.0 if true in rank_arr[:10] else 0.0)
        ndcg10_vals.append(ndcg_at_k(rank_arr, true, k=10))

        # MRR: reciprocal rank (unbounded)
        pos = np.where(rank_arr == true)[0]
        mrr_vals.append(1.0 / (pos[0] + 1) if len(pos) > 0 else 0.0)

    return {
        "hr1":    float(np.mean(hr1_vals)),
        "hr5":    float(np.mean(hr5_vals)),
        "hr10":   float(np.mean(hr10_vals)),
        "ndcg10": float(np.mean(ndcg10_vals)),
        "mrr":    float(np.mean(mrr_vals)),
        "n":      n,
    }


cell_end("CELL 08-04", t0)


[CELL 08-04] Define evaluation metrics
[CELL 08-04] start=2026-04-09T14:28:04
[CELL 08-04] elapsed=0.00s  done


In [6]:
# [CELL 08-05] GRURecommender — same architecture as NB06/NB07

t0 = cell_start("CELL 08-05", "Define GRURecommender")


class GRURecommender(nn.Module):
    """GRU-based sequential recommender.

    Architecture (matches NB06/NB07 — LOCKED):
        Embedding(n_items, embedding_dim) -> GRU(embedding_dim, hidden_dim) -> Linear(hidden_dim, n_items)
    """

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_items      = n_items
        self.embedding_dim = embedding_dim
        self.hidden_dim   = hidden_dim
        self.num_layers   = num_layers

        self.embedding = nn.Embedding(n_items + 1, embedding_dim, padding_idx=0)
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=gru_dropout
        )
        self.fc = nn.Linear(hidden_dim, n_items)

    def forward(self, seq: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        Args:
            seq:     (batch, max_len) padded item-id sequences
            lengths: (batch,) actual sequence lengths
        Returns:
            logits: (batch, n_items)
        """
        emb = self.embedding(seq)                       # (B, L, E)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)                       # h_n: (layers, B, H)
        h_last = h_n[-1]                                # (B, H)
        return self.fc(h_last)                          # (B, n_items)


# Quick sanity check
_m = GRURecommender(n_items=100, **GRU_CFG)
_x = torch.randint(1, 100, (4, 10))
_l = torch.tensor([10, 8, 5, 3])
_o = _m(_x, _l)
assert _o.shape == (4, 100), f"Expected (4,100) got {_o.shape}"
del _m, _x, _l, _o

cell_end("CELL 08-05", t0)


[CELL 08-05] Define GRURecommender
[CELL 08-05] start=2026-04-09T14:28:04
[CELL 08-05] elapsed=0.05s  done


In [7]:
# [CELL 08-06] functional_forward — uses torch.func.functional_call (fast CUDA GRU)

t0 = cell_start("CELL 08-06", "Define functional_forward for FOMAML")

from torch.func import functional_call

def functional_forward(
    model: GRURecommender,
    params: Dict[str, torch.Tensor],
    seqs: torch.Tensor,
    lengths: torch.Tensor,
) -> torch.Tensor:
    """
    Functional forward using torch.func.functional_call.
    Runs the model native optimized CUDA GRU with the given params dict.
    Vastly faster than manual step-by-step GRU loop.
    """
    return functional_call(model, params, (seqs, lengths))


cell_end("CELL 08-06", t0)



[CELL 08-06] Define functional_forward for FOMAML
[CELL 08-06] start=2026-04-09T14:28:04
[CELL 08-06] elapsed=0.00s  done


In [8]:
# [CELL 08-07] pad_sequences + get_episode_data helpers

t0 = cell_start("CELL 08-07", "Define pad_sequences + get_episode_data")


def pad_sequences(
    sequences: List[List[int]],
    max_len: int,
    pad_value: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pad a list of variable-length sequences to max_len.

    Returns:
        padded:  (batch, max_len) LongTensor
        lengths: (batch,)         LongTensor of actual lengths (clipped to max_len)
    """
    batch = len(sequences)
    padded = torch.full((batch, max_len), pad_value, dtype=torch.long)
    lengths = torch.zeros(batch, dtype=torch.long)
    for i, seq in enumerate(sequences):
        seq = seq[:max_len]
        l   = len(seq)
        if l > 0:
            padded[i, :l] = torch.tensor(seq, dtype=torch.long)
        lengths[i] = max(l, 1)   # avoid zero-length (GRU pack requirement)
    return padded, lengths


def get_episode_data(
    episode: pd.Series,
    pairs_idx: pd.DataFrame,
    max_seq_len: int = MAX_SEQ_LEN,
    device: str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Fetch and tensorise support/query batches for one episode.

    The pairs table must have columns:
        prefix_ids (list[int] or stringified list), label_id (int)

    Returns:
        sup_seqs, sup_lengths, sup_labels  — support batch
        qry_seqs, qry_lengths, qry_labels  — query batch
    """
    sup_ids = episode["support_pair_ids"]
    qry_ids = episode["query_pair_ids"]

    if isinstance(sup_ids, str):
        import ast
        sup_ids = ast.literal_eval(sup_ids)
        qry_ids = ast.literal_eval(qry_ids)

    def fetch_batch(pair_ids):
        rows  = pairs_idx.loc[pair_ids]
        seqs  = []
        for prefix in rows["prefix"]:
            if isinstance(prefix, str):
                import ast
                prefix = ast.literal_eval(prefix)
            seqs.append(list(prefix))
        labels = rows["label"].tolist()
        return seqs, labels

    sup_seqs_list, sup_labels_list = fetch_batch(sup_ids)
    qry_seqs_list, qry_labels_list = fetch_batch(qry_ids)

    sup_seqs, sup_lengths = pad_sequences(sup_seqs_list, max_seq_len)
    qry_seqs, qry_lengths = pad_sequences(qry_seqs_list, max_seq_len)

    sup_labels = torch.tensor(sup_labels_list, dtype=torch.long)
    qry_labels = torch.tensor(qry_labels_list, dtype=torch.long)

    return (
        sup_seqs.to(device),  sup_lengths.to(device),  sup_labels.to(device),
        qry_seqs.to(device),  qry_lengths.to(device),  qry_labels.to(device),
    )


cell_end("CELL 08-07", t0)


[CELL 08-07] Define pad_sequences + get_episode_data
[CELL 08-07] start=2026-04-09T14:28:04
[CELL 08-07] elapsed=0.00s  done


In [9]:
# [CELL 08-08] Load pretrained weights — warm-start initialisation
#
# This cell loads the GRU backbone trained on the full training set (NB06).
# The pretrained weights provide a warm-start for meta-learning.
# outer_lr=0.0001 (10x lower than NB07) preserves this initialisation.

t0 = cell_start("CELL 08-08", "Load pretrained weights (warm-start init)")

seed_everything(GLOBAL_SEED)   # deterministic

# ── Build meta-model ──────────────────────────────────────────
meta_model = GRURecommender(
    n_items=n_items,
    embedding_dim=GRU_CFG["embedding_dim"],
    hidden_dim=GRU_CFG["hidden_dim"],
    num_layers=GRU_CFG["num_layers"],
    dropout=GRU_CFG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in meta_model.parameters())
print(f"[CELL 08-08] Meta-model parameters: {n_params:,}")

# ── Warm-start: load pretrained GRU backbone ─────────────────
print(f"[CELL 08-08] Loading pretrained weights from: {PRETRAINED_PATH}")
pretrained_state = torch.load(PRETRAINED_PATH, map_location=DEVICE)
meta_model.load_state_dict(pretrained_state)
print("[CELL 08-08] Pretrained weights loaded successfully.")
print("[CELL 08-08] NOTE: inner loop uses STANDARD uniform steps — no SRS modifications.")
print(f"[CELL 08-08] NOTE: outer_lr={OUTER_LR} (10x lower than NB07) to preserve pretrained init.")

cell_end("CELL 08-08", t0)


[CELL 08-08] Load pretrained weights (warm-start init)
[CELL 08-08] start=2026-04-09T14:28:05
[CELL 08-08] Meta-model parameters: 367,341
[CELL 08-08] Loading pretrained weights from: /home/user/jamalla/anonymous-users-mooc-session-meta/models/baselines/gru_global_xuetangx.pth
[CELL 08-08] Pretrained weights loaded successfully.
[CELL 08-08] NOTE: inner loop uses STANDARD uniform steps — no SRS modifications.
[CELL 08-08] NOTE: outer_lr=0.0001 (10x lower than NB07) to preserve pretrained init.
[CELL 08-08] elapsed=0.40s  done


In [10]:
# [CELL 08-09] Warm-Start MAML training loop
#
# Differences from NB07 Standard MAML:
#   1. meta_model initialized from pretrained gru_global_xuetangx.pth  (cell 08-08)
#   2. OUTER_LR = 0.0001  (10x lower than NB07's 0.001)
#   3. Inner loop: STANDARD uniform gradient steps — NO SRS weighting
#
# Everything else (ALPHA_BASE, NUM_INNER_STEPS, BATCH_SIZE, N_ITERATIONS) unchanged.

t0_train = cell_start(
    "CELL 08-09", "Warm-Start FOMAML training (standard inner loop)",
    n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
    alpha_base=ALPHA_BASE, inner_steps=NUM_INNER_STEPS,
    outer_lr=OUTER_LR, warmstart=True, srs_adaptive=False
)

meta_optimizer = torch.optim.Adam(meta_model.parameters(), lr=OUTER_LR)

# ── Prepare training index ────────────────────────────────────
train_indices = list(range(len(episodes_train)))

# ── Tracking ─────────────────────────────────────────────────
train_losses  = []
val_hr10_hist = []
best_val_hr10 = -1.0
best_iter     = 0
best_state    = None

# ── Outer loop ───────────────────────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    meta_model.train()
    meta_optimizer.zero_grad()

    # Sample batch of task indices
    batch_idx = np.random.choice(train_indices, size=min(BATCH_SIZE, len(train_indices)), replace=False)
    batch_episodes = episodes_train.iloc[batch_idx]

    outer_loss = torch.tensor(0.0, device=DEVICE)
    n_valid    = 0

    for _, episode in batch_episodes.iterrows():
        try:
            sup_seqs, sup_lengths, sup_labels, \
            qry_seqs, qry_lengths, qry_labels = get_episode_data(
                episode, pairs_train_idx, MAX_SEQ_LEN, DEVICE
            )
        except (KeyError, ValueError):
            continue

        # ── Inner loop (FOMAML: create_graph=False) ──────────
        # Standard uniform gradient steps — NO SRS modifications
        params = {n: p.clone() for n, p in meta_model.named_parameters()}

        for _step in range(NUM_INNER_STEPS):
            logits = functional_forward(meta_model, params, sup_seqs, sup_lengths)
            loss_s = F.cross_entropy(logits, sup_labels)
            grads  = torch.autograd.grad(
                loss_s, list(params.values()), create_graph=False
            )
            params = {
                n: p - ALPHA_BASE * g
                for (n, p), g in zip(params.items(), grads)
            }

        # ── Query loss (outer loss) ───────────────────────────
        # Standard sum of query losses — NO SRS weighting
        qry_logits = functional_forward(meta_model, params, qry_seqs, qry_lengths)
        loss_q     = F.cross_entropy(qry_logits, qry_labels)
        outer_loss = outer_loss + loss_q
        n_valid   += 1

    if n_valid == 0:
        continue

    (outer_loss / n_valid).backward()
    torch.nn.utils.clip_grad_norm_(meta_model.parameters(), max_norm=5.0)
    meta_optimizer.step()

    train_losses.append(float(outer_loss.item() / n_valid))

    # ── Validation every VAL_EVERY iterations ────────────────
    if iteration % VAL_EVERY == 0:
        meta_model.train()  # keep train mode — inner loop needs cuDNN backward
        val_scores_all = []
        val_labels_all = []

        for _, ep in episodes_val.iterrows():
            try:
                sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
                    ep, pairs_val_idx, MAX_SEQ_LEN, DEVICE
                )
            except (KeyError, ValueError):
                continue

            # Adapt on support — params_v must require grad, so clone OUTSIDE no_grad

            params_v = {n: p.clone() for n, p in meta_model.named_parameters()}
            for _step in range(NUM_INNER_STEPS):
                lg_v   = functional_forward(meta_model, params_v, sup_s, sup_l)
                ls_v   = F.cross_entropy(lg_v, sup_lb)
                gv     = torch.autograd.grad(ls_v, list(params_v.values()), create_graph=False)
                params_v = {n: p - ALPHA_BASE * g for (n, p), g in zip(params_v.items(), gv)}

            with torch.no_grad():

                qry_lg = functional_forward(meta_model, params_v, qry_s, qry_l)
            val_scores_all.append(qry_lg.detach().cpu().numpy())
            val_labels_all.extend(qry_lb.cpu().tolist())

        if len(val_labels_all) > 0:
            val_scores_np = np.vstack(val_scores_all)
            val_labels_np = np.array(val_labels_all)
            val_m = compute_metrics(val_scores_np, val_labels_np)
            val_hr10_hist.append((iteration, val_m["hr10"]))

            if val_m["hr10"] > best_val_hr10:
                best_val_hr10 = val_m["hr10"]
                best_iter     = iteration
                best_state    = copy.deepcopy(meta_model.state_dict())
                torch.save(best_state, CHECKPOINT_PATH)

            print(
                f"[CELL 08-09] iter={iteration:4d} "
                f"train_loss={train_losses[-1]:.4f}  "
                f"val_HR@10={val_m['hr10']*100:.2f}%  "
                f"best_HR@10={best_val_hr10*100:.2f}%@{best_iter}"
            )

print(f"\n[CELL 08-09] Training complete.")
print(f"[CELL 08-09] Best val HR@10: {best_val_hr10*100:.2f}% at iteration {best_iter}")
print(f"[CELL 08-09] Checkpoint saved: {CHECKPOINT_PATH}")

cell_end("CELL 08-09", t0_train, best_iter=best_iter, best_val_hr10=f"{best_val_hr10*100:.2f}%")



[CELL 08-09] Warm-Start FOMAML training (standard inner loop)
[CELL 08-09] start=2026-04-09T14:28:05
[CELL 08-09] n_iterations=3000
[CELL 08-09] batch_size=32
[CELL 08-09] alpha_base=0.01
[CELL 08-09] inner_steps=5
[CELL 08-09] outer_lr=0.0001
[CELL 08-09] warmstart=True
[CELL 08-09] srs_adaptive=False
[CELL 08-09] iter= 100 train_loss=3.0573  val_HR@10=54.51%  best_HR@10=54.51%@100
[CELL 08-09] iter= 200 train_loss=3.3359  val_HR@10=54.62%  best_HR@10=54.62%@200
[CELL 08-09] iter= 300 train_loss=3.1730  val_HR@10=54.71%  best_HR@10=54.71%@300
[CELL 08-09] iter= 400 train_loss=2.5957  val_HR@10=54.65%  best_HR@10=54.71%@300
[CELL 08-09] iter= 500 train_loss=3.3655  val_HR@10=54.67%  best_HR@10=54.71%@300
[CELL 08-09] iter= 600 train_loss=3.2605  val_HR@10=54.73%  best_HR@10=54.73%@600
[CELL 08-09] iter= 700 train_loss=3.1257  val_HR@10=54.64%  best_HR@10=54.73%@600
[CELL 08-09] iter= 800 train_loss=3.1806  val_HR@10=54.67%  best_HR@10=54.73%@600
[CELL 08-09] iter= 900 train_loss=3.031

In [11]:
# [CELL 08-10] Test evaluation — load best checkpoint, evaluate on test episodes

t0 = cell_start("CELL 08-10", "Test evaluation (best checkpoint)")

# Load best checkpoint
if not CHECKPOINT_PATH.exists():
    raise RuntimeError(f"Checkpoint missing: {CHECKPOINT_PATH}. Run training cell first.")

meta_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
meta_model.train()  # keep train mode — inner loop needs cuDNN backward
print(f"[CELL 08-10] Loaded checkpoint from iter {best_iter}: {CHECKPOINT_PATH}")

test_scores_all = []
test_labels_all = []
n_skipped = 0

for _, ep in episodes_test.iterrows():
    try:
        sup_s, sup_l, sup_lb, qry_s, qry_l, qry_lb = get_episode_data(
            ep, pairs_test_idx, MAX_SEQ_LEN, DEVICE
        )
    except (KeyError, ValueError):
        n_skipped += 1
        continue

    # Adapt on support set (K=5, standard inner loop)
    params_t = {n: p.clone() for n, p in meta_model.named_parameters()}
    for _step in range(NUM_INNER_STEPS):
        lg_t = functional_forward(meta_model, params_t, sup_s, sup_l)
        ls_t = F.cross_entropy(lg_t, sup_lb)
        gt   = torch.autograd.grad(ls_t, list(params_t.values()), create_graph=False)
        params_t = {n: p - ALPHA_BASE * g for (n, p), g in zip(params_t.items(), gt)}

    with torch.no_grad():
        qry_lg = functional_forward(meta_model, params_t, qry_s, qry_l)
    test_scores_all.append(qry_lg.detach().cpu().numpy())
    test_labels_all.extend(qry_lb.cpu().tolist())

print(f"[CELL 08-10] episodes evaluated: {len(test_scores_all)}, skipped: {n_skipped}")

test_scores_np = np.vstack(test_scores_all)
test_labels_np = np.array(test_labels_all)

test_metrics = compute_metrics(test_scores_np, test_labels_np)

test_hr1    = test_metrics["hr1"]
test_hr5    = test_metrics["hr5"]
test_hr10   = test_metrics["hr10"]
test_ndcg10 = test_metrics["ndcg10"]
test_mrr    = test_metrics["mrr"]

print(f"\n[CELL 08-10] TEST RESULTS (K={K_SUPPORT} support, Q={Q_QUERY} query):")
print(f"  HR@1    = {test_hr1*100:.2f}%")
print(f"  HR@5    = {test_hr5*100:.2f}%")
print(f"  HR@10   = {test_hr10*100:.2f}%   <- PRIMARY")
print(f"  NDCG@10 = {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"  MRR     = {test_mrr*100:.2f}%")

cell_end("CELL 08-10", t0, n_test_samples=len(test_labels_all))


[CELL 08-10] Test evaluation (best checkpoint)
[CELL 08-10] start=2026-04-09T15:10:28
[CELL 08-10] Loaded checkpoint from iter 1600: /home/user/jamalla/anonymous-users-mooc-session-meta/models/maml/maml_warmstart_xuetangx.pth
[CELL 08-10] episodes evaluated: 986, skipped: 0

[CELL 08-10] TEST RESULTS (K=5 support, Q=10 query):
  HR@1    = 26.53%
  HR@5    = 45.09%
  HR@10   = 53.51%   <- PRIMARY
  NDCG@10 = 39.07%   <- PRIMARY
  MRR     = 35.70%
[CELL 08-10] n_test_samples=9860
[CELL 08-10] elapsed=18.78s  done


In [12]:
# [CELL 08-11] Standard result block (CLAUDE.md mandatory format)

print("=" * 55)
print(f"RESULTS — {NOTEBOOK_NAME} — {DATASET}")
print("=" * 55)
print(f"Protocol:      K={K_SUPPORT} support, Q={Q_QUERY} query")
print(f"Test episodes: {len(episodes_test)}")
print(f"Seed:          {GLOBAL_SEED}")
print()
print(f"HR@1:          {test_hr1*100:.2f}%")
print(f"HR@5:          {test_hr5*100:.2f}%")
print(f"HR@10:         {test_hr10*100:.2f}%   <- PRIMARY")
print(f"NDCG@10:       {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"MRR:           {test_mrr*100:.2f}%")
print()
print(f"Best val iter: {best_iter}")
print(f"Best val HR@10:{best_val_hr10*100:.2f}%")
print("=" * 55)
print()
print("CONFIG DIFFERENCES FROM NB07 (Standard MAML):")
print(f"  outer_lr      = {OUTER_LR}  (NB07: 0.001 — 10x lower here)")
print(f"  warm_start    = True  (loaded gru_global_{DATASET}.pth)")
print(f"  srs_adaptive  = False (standard uniform inner loop)")

RESULTS — 08_warmstart_maml_xuetangx — xuetangx
Protocol:      K=5 support, Q=10 query
Test episodes: 986
Seed:          20260107

HR@1:          26.53%
HR@5:          45.09%
HR@10:         53.51%   <- PRIMARY
NDCG@10:       39.07%   <- PRIMARY
MRR:           35.70%

Best val iter: 1600
Best val HR@10:54.76%

CONFIG DIFFERENCES FROM NB07 (Standard MAML):
  outer_lr      = 0.0001  (NB07: 0.001 — 10x lower here)
  warm_start    = True  (loaded gru_global_xuetangx.pth)
  srs_adaptive  = False (standard uniform inner loop)


In [13]:
# [CELL 08-12] Save results JSON

t0 = cell_start("CELL 08-12", "Save results JSON")

results = {
    "run_id":        RUN_ID,
    "notebook":      NOTEBOOK_NAME,
    "dataset":       DATASET,
    "run_tag":       RUN_TAG,
    "created_at":    datetime.now().isoformat(timespec="seconds"),
    "config": {
        "K_support":         K_SUPPORT,
        "Q_query":           Q_QUERY,
        "alpha_base":        ALPHA_BASE,
        "num_inner_steps":   NUM_INNER_STEPS,
        "outer_lr":          OUTER_LR,
        "batch_size":        BATCH_SIZE,
        "n_iterations":      N_ITERATIONS,
        "global_seed":       GLOBAL_SEED,
        "gru_cfg":           GRU_CFG,
        "warmstart":         True,
        "srs_adaptive":      False,
        "pretrained_path":   str(PRETRAINED_PATH),
    },
    "test_metrics": {
        "hr1":    test_hr1,
        "hr5":    test_hr5,
        "hr10":   test_hr10,
        "ndcg10": test_ndcg10,
        "mrr":    test_mrr,
        "n":      int(len(test_labels_all)),
    },
    "val_best": {
        "best_iter":     best_iter,
        "best_val_hr10": best_val_hr10,
    },
    "checkpoint": str(CHECKPOINT_PATH),
}

write_json_atomic(RESULTS_PATH, results)
print(f"[CELL 08-12] Results saved: {RESULTS_PATH}")

# Update report.json
report = read_json(REPORT_PATH)
report["metrics"] = results["test_metrics"]
report["key_findings"] = [
    f"Warm-Start MAML ablation: HR@10={test_hr10*100:.2f}%, NDCG@10={test_ndcg10*100:.2f}%",
    f"Best val HR@10={best_val_hr10*100:.2f}% at iteration {best_iter}",
    f"Hyperparams: alpha={ALPHA_BASE}, inner_steps={NUM_INNER_STEPS}, outer_lr={OUTER_LR} (warm-start), batch={BATCH_SIZE}, iters={N_ITERATIONS}",
    f"Warm-start from: {PRETRAINED_PATH}",
    f"Inner loop: standard uniform (no SRS-Adaptive) — ablation only",
    f"Test episodes evaluated: {len(test_labels_all)}, skipped: {n_skipped}",
]
write_json_atomic(REPORT_PATH, report)

# Update manifest.json
manifest = read_json(MANIFEST_PATH)
manifest["artifacts"].extend([
    {"type": "results_json",  "path": str(RESULTS_PATH)},
    {"type": "checkpoint",    "path": str(CHECKPOINT_PATH)},
    {"type": "report",        "path": str(REPORT_PATH)},
    {"type": "config",        "path": str(CONFIG_PATH)},
])
write_json_atomic(MANIFEST_PATH, manifest)

cell_end("CELL 08-12", t0)


[CELL 08-12] Save results JSON
[CELL 08-12] start=2026-04-09T15:10:47
[CELL 08-12] Results saved: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/08_warmstart_maml_xuetangx/20260409_142803/results_08_warmstart_maml_xuetangx.json
[CELL 08-12] elapsed=0.02s  done


In [14]:
# [CELL 08-13] Update PROJECT_STATE.md with results

t0 = cell_start("CELL 08-13", "Update PROJECT_STATE.md")

state_path = PATHS["PROJECT_STATE"]

result_block = (
    f"\n\n## {NOTEBOOK_NAME} — Results\n"
    f"Run: {RUN_TAG}  |  Seed: {GLOBAL_SEED}\n"
    f"Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query | Test episodes: {len(episodes_test)}\n"
    f"Ablation: Warm-Start MAML (no SRS-Adaptive) | outer_lr={OUTER_LR}\n"
    f"\n"
    f"| Metric   | Value    |\n"
    f"|----------|----------|\n"
    f"| HR@1     | {test_hr1*100:.2f}%  |\n"
    f"| HR@5     | {test_hr5*100:.2f}%  |\n"
    f"| HR@10    | {test_hr10*100:.2f}%  |\n"
    f"| NDCG@10  | {test_ndcg10*100:.2f}%  |\n"
    f"| MRR      | {test_mrr*100:.2f}%  |\n"
    f"\nBest val HR@10: {best_val_hr10*100:.2f}% @ iter {best_iter}\n"
    f"Checkpoint: {CHECKPOINT_PATH}\n"
)

if state_path.exists():
    existing = state_path.read_text(encoding="utf-8")
    marker = f"## {NOTEBOOK_NAME}"
    if marker in existing:
        # Replace existing block
        lines = existing.split("\n")
        new_lines = []
        skip = False
        for line in lines:
            if line.startswith(marker):
                skip = True
            elif skip and line.startswith("## ") and not line.startswith(marker):
                skip = False
            if not skip:
                new_lines.append(line)
        updated = "\n".join(new_lines) + result_block
    else:
        updated = existing + result_block
    state_path.write_text(updated, encoding="utf-8")
    print(f"[CELL 08-13] PROJECT_STATE.md updated: {state_path}")
else:
    state_path.write_text(result_block, encoding="utf-8")
    print(f"[CELL 08-13] PROJECT_STATE.md created: {state_path}")

cell_end("CELL 08-13", t0)


[CELL 08-13] Update PROJECT_STATE.md
[CELL 08-13] start=2026-04-09T15:10:47
[CELL 08-13] PROJECT_STATE.md updated: /home/user/jamalla/anonymous-users-mooc-session-meta/PROJECT_STATE.md
[CELL 08-13] elapsed=0.02s  done


## Notebook 08 Complete — Warm-Start MAML Ablation (XuetangX)

**What was done:**
- Loaded pretrained GRU backbone from `models/baselines/gru_global_xuetangx.pth` (NB06 output)
- Applied FOMAML with lower outer_lr=0.0001 to preserve pretrained knowledge
- Inner loop: standard uniform gradient steps — **NO SRS modifications**
- Outer loop: standard sum of query losses — **NO SRS weighting**
- Validation every 100 iterations; best HR@10 checkpoint saved at iter **1600**

**Config differences from NB07 (Standard MAML):**
```
outer_lr   = 0.0001   ← 10x lower (NB07 uses 0.001)
warm_start = True     ← loaded pretrained weights
srs_adaptive = False  ← standard inner loop (ablation)
```

**Test Results (K=5 support, Q=10 query, 986 episodes):**
| Metric | Value |
|--------|-------|
| HR@1 | 26.53% |
| HR@5 | 45.09% |
| **HR@10** | **53.51%** |
| **NDCG@10** | **39.07%** |
| MRR | 35.70% |

→ Warm-start alone improves substantially over Standard MAML (NB07: 47.46% → NB08: 53.51% HR@10).

**Next:** `09_srs_validation_xuetangx.ipynb` (validate SRS formula before NB10/NB11)
